In [1]:
from playwright.async_api import async_playwright
import pandas as pd
import asyncio
import re
from pathlib import Path

In [2]:
URL = "https://pcm2019.shinyapps.io/XDeathDB/"

In [3]:
# Create directories
PROJECT_ROOT = Path("..")

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
XDEATHDB_DATA = RAW_DATA / "XDeathDB"

RAW_DATA.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
XDEATHDB_DATA.mkdir(parents=True, exist_ok=True)

In [7]:
async def extract_xdeathdb():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page(viewport={"width": 1400, "height": 1000})

        # Open XDeathDB
        await page.goto(
            URL,
            wait_until="domcontentloaded",
            timeout=60000
        )
        await page.wait_for_timeout(5000)

        # Open Network Analysis
        await page.get_by_role(
            "link",
            name="Network Analysis"
        ).click()
        await page.wait_for_timeout(5000)

        # Locate the main controls
        cdm_input = page.locator(
            "#tabset1subitem1-tab4-pathinput-cdm-selectized"
        )

        network_input = page.locator(
            "#tabset1subitem1-tab4-pathinput-network-selectized"
        )

        disease_input = page.locator(
            "#tabset1subitem1-tab4-pathinput-c2-selectized"
        )

        cell_death_modes = [
            "Miotic CD",
            "Autophagy",
            "Autosis",
            "Immunogenic",
            "Efferocytosis",
            "Ferroptosis",
            "MPT",
            "Necroptosis",
            "Inrinsic",
            "Parthanos",
            "Pyroptosis",
            "Lysozomal"
        ]

        # Process every cell death mode
        for cell_death_mode in cell_death_modes:

            print(f"\nProcessing: {cell_death_mode}")

            # Select cell death mode
            await cdm_input.evaluate(
                """
                el => {
                    el.scrollIntoView({
                        block: 'center',
                        inline: 'center'
                    });
                    el.click();
                }
                """
            )
            await page.wait_for_timeout(500)

            cdm_options = page.locator(
                ".selectize-dropdown .option"
            )

            selected_cdm = None

            for i in range(await cdm_options.count()):

                candidate = cdm_options.nth(i)

                if await candidate.is_visible():

                    text = (await candidate.inner_text()).strip()

                    if text.lower() == cell_death_mode.lower():
                        selected_cdm = candidate
                        break

            if selected_cdm is None:
                print(
                    f"Could not find cell death mode: {cell_death_mode}"
                )
                continue

            await selected_cdm.click(force=True)
            await page.wait_for_timeout(2000)

            # Open network dropdown
            await network_input.click(force=True)
            await page.wait_for_timeout(500)

            # Find all currently available network options
            network_options = page.locator(
                ".selectize-dropdown .option"
            )

            available_networks = []

            for i in range(await network_options.count()):

                candidate = network_options.nth(i)

                if await candidate.is_visible():

                    text = (await candidate.inner_text()).strip()

                    if text.startswith("Network "):
                        available_networks.append(text)

            print(
                f"Available networks: {available_networks}"
            )

            # Select every network that actually exists
            for network in available_networks:

                # Reopen the dropdown after each selection
                await network_input.click(force=True)
                await page.wait_for_timeout(300)

                options = page.locator(
                    ".selectize-dropdown .option"
                )

                selected_network = None

                for i in range(await options.count()):

                    candidate = options.nth(i)

                    if await candidate.is_visible():

                        text = (await candidate.inner_text()).strip()

                        if text == network:
                            selected_network = candidate
                            break

                if selected_network is not None:

                    print(f"Selecting: {network}")

                    await selected_network.click(force=True)
                    await page.wait_for_timeout(300)

            print(
                f"Selected {len(available_networks)} networks."
            )

            # Open disease dropdown
            await disease_input.evaluate(
                """
                el => {
                    el.scrollIntoView({
                        block: 'center',
                        inline: 'center'
                    });
                    el.click();
                }
                """
            )

            await page.wait_for_timeout(500)

            # Select "All" diseases
            disease_options = page.locator(
                ".selectize-dropdown .option"
            )

            all_option = None

            for i in range(await disease_options.count()):

                candidate = disease_options.nth(i)

                if await candidate.is_visible():

                    text = (await candidate.inner_text()).strip()

                    if text == "All":
                        all_option = candidate
                        break

            if all_option is None:
                print(
                    "Could not find 'All' disease option."
                )
                continue

            await all_option.click(force=True)

            # Wait for genes to populate
            await page.wait_for_timeout(8000)

            # Find all Selectize items
            all_items = page.locator(
                ".selectize-control .item[data-value]"
            )

            total_items = await all_items.count()

            # Extract genes after "regulation of binding"
            genes = []
            start_extracting = False

            for i in range(total_items):

                item = all_items.nth(i)

                if not await item.is_visible():
                    continue

                value = await item.get_attribute(
                    "data-value"
                )

                text = (await item.inner_text()).strip()

                if start_extracting:

                    if value:
                        genes.append(value)

                elif text == "regulation of binding":

                    start_extracting = True

            # Remove duplicates while preserving order
            genes = list(dict.fromkeys(genes))

            # Save genes
            filename = (
                XDEATHDB_DATA /
                f"{cell_death_mode}.txt"
            )

            with open(
                filename,
                "w",
                encoding="utf-8"
            ) as f:

                for gene in genes:
                    f.write(gene + "\n")

            print(
                f"Saved {len(genes)} genes to {filename}"
            )

        await browser.close()

await extract_xdeathdb()


Processing: Miotic CD
Could not find cell death mode: Miotic CD

Processing: Autophagy
Available networks: ['Network 1', 'Network 2', 'Network 3', 'Network 4', 'Network 5', 'Network 6', 'Network 7', 'Network 8']
Selecting: Network 1
Selecting: Network 2
Selecting: Network 3
Selecting: Network 4
Selecting: Network 5
Selecting: Network 6
Selecting: Network 7
Selecting: Network 8
Selected 8 networks.
Saved 278 genes to ../data/raw/XDeathDB/Autophagy.txt

Processing: Autosis
Available networks: ['Network 1', 'Network 2', 'Network 3', 'Network 4', 'Network 5', 'Network 6', 'Network 7', 'Network 8', 'Network 9', 'Network 10', 'Network 11', 'Network 12', 'Network 13', 'Network 14', 'Network 15', 'Network 16', 'Network 17', 'Network 18', 'Network 19', 'Network 20', 'Network 21', 'Network 22', 'Network 23', 'Network 24', 'Network 25']
Selecting: Network 1
Selecting: Network 2
Selecting: Network 3
Selecting: Network 4
Selecting: Network 5
Selecting: Network 6
Selecting: Network 7
Selecting: Net

In [10]:
# Find all XDeathDB text files
txt_files = list(XDEATHDB_DATA.glob("*.txt"))

# Store genes for each cell death mode
mode_genes = {}

for file in txt_files:
    mode = file.stem

    with open(file, "r", encoding="utf-8") as f:
        genes = [
            line.strip()
            for line in f
            if line.strip()
        ]

    # Remove duplicates while preserving order
    mode_genes[mode] = list(dict.fromkeys(genes))

# Get all unique genes
all_genes = sorted(
    set(
        gene
        for genes in mode_genes.values()
        for gene in genes
    )
)

In [11]:
# Create binary Gene × Cell Death Mode table
gene_table = pd.DataFrame(
    0,
    index=all_genes,
    columns=mode_genes.keys()
)

# Mark gene presence
for mode, genes in mode_genes.items():
    gene_table.loc[genes, mode] = 1

# Name the index
gene_table.index.name = "Gene"

# Add final hits column
gene_table["Final Hits"] = gene_table.sum(axis=1)

# Sort genes by Final Hits in descending order
gene_table = gene_table.sort_values(
    by="Final Hits",
    ascending=False
)

# Save CSV
output_file = PROCESSED_DATA / "XDeathDB_gene_cell_death_matrix.csv"
gene_table.to_csv(output_file)

print(f"Saved sorted matrix to: {output_file}")

Saved sorted matrix to: ../data/processed/XDeathDB_gene_cell_death_matrix.csv
